# AR Completion — Adversarial Reactivity for explain_func metrics

Closes the meta-validation gap reported in the thesis: the 2026-06 full run
returned `ar_score = NaN` for **max_sensitivity**, **avg_sensitivity**, and
**random_logit**, because those metrics ignore the provided `a_batch`
(they re-compute attributions internally via `explain_func`), so degrading
the attribution array left their scores constant across all AR levels.

The fix (`make_degraded_explain_func` + `degrade_explain_func=True`,
commit on `master`) injects the degradation into the explanation function
itself. This notebook re-runs **only the missing AR legs**
(2 models x 7 FAE x 3 metrics = 42 cells), reusing everything already
stashed in Drive (weights, cached test split, reliability CSV), and patches
`meta_evaluation_reliability.csv` in place (backing up the original).

**Prerequisites:**
1. `git push` the local master first — this notebook clones from GitHub and
   needs the `degrade_explain_func` commit.
2. Runtime: GPU (T4). Estimated total: ~1–2 h; progress is checkpointed to
   Drive after every cell, so a runtime reset resumes where it left off.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_THESIS = '/content/drive/MyDrive/thesis'

import os
os.makedirs(f'{DRIVE_THESIS}/results', exist_ok=True)
print(f'Drive mounted. Thesis folder: {DRIVE_THESIS}')

## 2. Clone Repository (private — needs the GH_TOKEN Colab secret)

In [ ]:
import os

# Private repo: read the GitHub token from Colab secrets (key icon in the
# left sidebar -> add secret named GH_TOKEN with "Notebook access" enabled).
from google.colab import userdata
try:
    GH_TOKEN = userdata.get('GH_TOKEN')
    CLONE_URL = f'https://{GH_TOKEN}@github.com/dawkopagh/fae-metrics-master-thesis.git'
except Exception:
    print('No GH_TOKEN secret found - trying anonymous clone (works only if public).')
    CLONE_URL = 'https://github.com/dawkopagh/fae-metrics-master-thesis.git'

# Clone the THESIS repo root into an unambiguous path; the code lives in its
# fae-metrics-master-thesis/ subdirectory.
THESIS_ROOT = '/content/thesis-repo'
CODE_DIR    = f'{THESIS_ROOT}/fae-metrics-master-thesis'

if os.path.exists(THESIS_ROOT):
    %cd {THESIS_ROOT}
    !git pull {CLONE_URL} master
else:
    !git clone {CLONE_URL} {THESIS_ROOT}

%cd {CODE_DIR}
print(f'Working directory: {os.getcwd()}')
print('HEAD commit:', end=' ')
!git rev-parse HEAD

# Guard: the AR fix must be present.
from pathlib import Path
assert 'make_degraded_explain_func' in Path(
    'src/meta_evaluation/metaquantus_wrapper.py').read_text(), (
    'AR-fix commit missing - run `git push` locally first, then re-run this cell.')
print('AR fix present.')

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q quantus==0.6.0

import captum, quantus, torch
print(f'captum  {captum.__version__} | quantus {quantus.__version__} | '
      f'torch {torch.__version__} | cuda {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 4. Weights from Drive (checksum-verified)

In [ ]:
RESNET_SHA     = '593dcb844b8359550e3d84667475bbd845f45a6cb388356480c0a55cb5173430'
SQUEEZENET_SHA = '8bbb43bbba4ee81e58e354295420bea33e55cfa2be11c31d82dce272d03b092d'

import os, hashlib
os.makedirs('weights', exist_ok=True)
!cp {DRIVE_THESIS}/weights/resnet18_isic2017.pth   weights/resnet18_isic2017.pth
!cp {DRIVE_THESIS}/weights/squeezenet_isic2017.pth weights/squeezenet_isic2017.pth

for path, expected in [('weights/resnet18_isic2017.pth', RESNET_SHA),
                       ('weights/squeezenet_isic2017.pth', SQUEEZENET_SHA)]:
    actual = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    status = 'OK' if actual == expected else 'MISMATCH!'
    print(f'{path}: {status}')
    assert actual == expected, f'Checksum mismatch for {path}'

## 5. Test Split from Drive (fallback: re-download)

In [ ]:
import os, sys
sys.path.insert(0, '.')

_needs_download = False
for split_dir in ['data/images/test', 'data/masks/test']:
    src = f'{DRIVE_THESIS}/{split_dir}'
    if os.path.exists(src):
        os.makedirs(split_dir, exist_ok=True)
        !cp -r {src}/* {split_dir}/
        n = sum(len(files) for _, _, files in os.walk(split_dir))
        print(f'{split_dir}: {n} files (from Drive)')
    else:
        print(f'{split_dir}: not found on Drive — will download')
        _needs_download = True

if _needs_download:
    from src.data.download_isic import download_isic2017
    download_isic2017(dest_dir='data', skip_existing=True)
print('Data scaffold ready.')

## 6. Load Reliability CSV (from the repo — the committed post-fix version)

Shows exactly which rows this notebook will fill in.

In [ ]:
import pandas as pd, numpy as np

REL_CSV = f'{THESIS_ROOT}/results/meta_evaluation_reliability.csv'
rel = pd.read_csv(REL_CSV)
print(f'{len(rel)} rows; statuses: {rel.status.value_counts().to_dict()}')

TARGET_METRICS = ['max_sensitivity', 'avg_sensitivity', 'random_logit']
todo = rel[rel.metric.isin(TARGET_METRICS)
           & (rel.status == 'completed')
           & rel.ar_score.isna()]
print(f'\nAR legs to complete: {len(todo)} '
      f'({todo.model.nunique()} models x {todo.fae_method.nunique()} FAE x '
      f'{todo.metric.nunique()} metrics)')
todo[['model', 'fae_method', 'metric', 'nr_score']].reset_index(drop=True)

## 7. Run the Missing AR Legs

Identical protocol to the 2026-06 meta-evaluation (same 12 test images,
targets from ResNet-18 predictions, per-model explain functions,
`n_levels=5`), with `degrade_explain_func=True`. Each finished cell is
checkpointed to Drive immediately.

In [ ]:
import time, numpy as np, torch, quantus, pandas as pd

from src.models.classifiers import load_resnet18, load_squeezenet
from src.data.isic_dataset  import ISIC2017Dataset
from src.pipeline           import _make_explain_func
from src.meta_evaluation.metaquantus_wrapper import adversarial_reactivity_test

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

models = {
    'resnet18':   load_resnet18('weights/resnet18_isic2017.pth',   device=device),
    'squeezenet': load_squeezenet('weights/squeezenet_isic2017.pth', device=device),
}

# Same 12 images + resnet18-predicted targets as the original meta-eval run.
dataset = ISIC2017Dataset(root_dir='data', split='test', image_size=224, return_mask=True)
N_IMAGES = 12
samples = [dataset[i] for i in range(N_IMAGES)]
images  = [s['image'] for s in samples]
targets = [
    int(models['resnet18'](s['image'].unsqueeze(0).to(device)).argmax(dim=1).item())
    for s in samples
]
imgs_np = [img.detach().cpu().numpy() for img in images]
x_stack = np.stack(imgs_np)
y_array = np.array(targets, dtype=np.int64)
print(f'{len(images)} test images, targets: {targets}')

FAE_NAMES = ['integrated_gradients', 'saliency', 'gradcam',
             'deep_lift', 'guided_backprop', 'lrp', 'occlusion']
# Per-model explain funcs (nested form — Grad-CAM target layer is
# architecture-specific; see commit 9117e7b).
fae_funcs = {
    m: {name: _make_explain_func(name, models[m], m, device) for name in FAE_NAMES}
    for m in models
}

# Production metric configs — identical to the original meta-eval notebook.
metric_fns = {
    'max_sensitivity': quantus.MaxSensitivity(
        nr_samples=10, lower_bound=0.2,
        normalise=False, abs=False, return_aggregate=False, disable_warnings=True),
    'avg_sensitivity': quantus.AvgSensitivity(
        nr_samples=10, lower_bound=0.2,
        normalise=False, abs=False, return_aggregate=False, disable_warnings=True),
    'random_logit': quantus.RandomLogit(
        num_classes=3, abs=True, normalise=True,
        return_aggregate=False, disable_warnings=True),
}

# Drive checkpoint: one row per finished (model, fae, metric); reruns resume.
CKPT = f'{DRIVE_THESIS}/results/ar_completion_progress.csv'
done = {}
import os
if os.path.exists(CKPT):
    ck = pd.read_csv(CKPT)
    done = {(r.model, r.fae_method, r.metric): float(r.ar_score)
            for r in ck.itertuples()}
    print(f'Resuming: {len(done)} cells already checkpointed.')

results = dict(done)
work = [(m, f, met) for m in models for f in FAE_NAMES for met in metric_fns]
work = [w for w in work if w not in done]
print(f'{len(work)} cells to run.\n')

for i, (model_name, fae_name, metric_name) in enumerate(work, 1):
    model = models[model_name]
    explain_fn = fae_funcs[model_name][fae_name]
    t0 = time.perf_counter()

    attrs_batch = explain_fn(model, x_stack, y_array)
    attributions = [np.asarray(attrs_batch[j]) for j in range(len(imgs_np))]

    ar = adversarial_reactivity_test(
        metric_fn=metric_fns[metric_name],
        model=model,
        images=imgs_np,
        attributions=attributions,
        targets=targets,
        explain_func=explain_fn,
        n_levels=5,
        # Cap at 0.9: at frac=1.0 the degraded explainer returns an
        # all-zero attribution, which Quantus rejects (the level would
        # just be dropped as NaN); 0.9 keeps all five levels informative.
        level_max=0.9,
        device=device,
        degrade_explain_func=True,   # the fix
    )
    elapsed = time.perf_counter() - t0
    key = (model_name, fae_name, metric_name)
    results[key] = ar['ar_score']

    row = pd.DataFrame([{'model': model_name, 'fae_method': fae_name,
                         'metric': metric_name, 'ar_score': ar['ar_score'],
                         'monotonicity': ar['monotonicity'],
                         'scores': ';'.join(f'{s:.6g}' for s in ar['scores']),
                         'runtime_seconds': round(elapsed, 1)}])
    row.to_csv(CKPT, mode='a', header=not os.path.exists(CKPT), index=False)

    print(f'[{i:2d}/{len(work)}] {model_name:10s} {fae_name:20s} '
          f'{metric_name:16s} AR={ar["ar_score"]:.3f} '
          f'(rho={ar["monotonicity"]:+.3f}) {elapsed:.0f}s')

print('\nAll AR legs complete.')

## 8. Patch the Reliability CSV and Save to Drive

Updates `ar_score` and `combined_reliability` (= 0.5 x (existing NR + new AR))
for the recomputed rows; the pre-patch file is backed up first.

In [ ]:
import shutil

rel = pd.read_csv(REL_CSV)
backup = f'{DRIVE_THESIS}/results/meta_evaluation_reliability_pre_AR.csv'
if not os.path.exists(backup):
    shutil.copy2(REL_CSV, backup)
    print(f'Backup of pre-AR CSV -> {backup}')

patched = 0
for (model_name, fae_name, metric_name), ar_score in results.items():
    m = ((rel.model == model_name) & (rel.fae_method == fae_name)
         & (rel.metric == metric_name))
    assert m.sum() == 1, f'Expected exactly 1 row for {(model_name, fae_name, metric_name)}'
    idx = rel.index[m][0]
    nr = float(rel.at[idx, 'nr_score'])
    rel.at[idx, 'ar_score'] = ar_score
    rel.at[idx, 'combined_reliability'] = (
        0.5 * (nr + ar_score) if not (np.isnan(nr) or np.isnan(ar_score))
        else float('nan'))
    patched += 1

print(f'Patched {patched} rows.')
print('Valid combined_reliability:',
      rel.combined_reliability.notna().sum(), '/', len(rel))

out_local = 'results/meta_evaluation_reliability.csv'
os.makedirs('results', exist_ok=True)
rel.to_csv(out_local, index=False)
out_drive = f'{DRIVE_THESIS}/results/meta_evaluation_reliability.csv'
shutil.copy2(out_local, out_drive)
print(f'Saved patched CSV -> {out_local} and {out_drive}')

print('''
NEXT STEPS (back on the local machine):
  1. Copy the patched CSV into the repo:
       cp <Drive>/thesis/results/meta_evaluation_reliability.csv results/
  2. Re-run the local aggregation chain (MQ-discount weights change):
       .venv/bin/python experiments/compare_rankings.py \
           --slice-csv ../results/full_run_7fae_12metrics_600.csv \
           --reliability-csv ../results/meta_evaluation_reliability.csv \
           --output-csv ../results/ranking_comparison.csv
       .venv/bin/python experiments/run_statistical_analysis.py \
           --ranking-csv ../results/ranking_comparison.csv \
           --tests-out ../results/statistical_tests.csv \
           --cd-out ../results/cd_summary.csv
       (from thesis root) python experiments/make_figures.py
       python experiments/render_results_tables.py --n-images-label "full run, 600 images"
  3. Reconcile the ch4/ch5/ch6 prose that cites the NR-only fallback and
     the 84/168 valid-rows count, then rebuild the PDF.''')